# LEVIR-Ship: INT8 PTQ root-cause isolation — H2 seed 42

**One model only:** `LEVIR-Ship/historical_reconstructions/RECON_H2_V1/seed_42/weights/best.pt`. Reuse the verified paper evaluators in `utils/ftsc_hf_all_trt_int8_levir_fp16parity_v2.py`.

Purpose: distinguish **calibrator CUDA buffer lifetime**, **Min-Max vs Entropy**, and **512-content→544 padding vs 544 resize during calibration**. Preprocessing at inference, NMS thresholds, test split, PT32 reference, and checkpoint stay fixed.

**Safety:** default `RUN_GPU=False`; no training, uploads, modifications to the production exporter, or writes outside this probe's fresh `runs/` directory. Each arm exports separate static 544 MAIN and 512 SMALL TensorRT INT8 engines; it never reuses the existing HF engines or cache. Do not run alongside heavy INT8 GPU jobs.

The notebook is written for the existing **Marimo Live Jupyter-compatible environment** after syncing the local `yolo_code`. It does not download datasets, install dependencies, or change Git branches. Dataset must already be prepared under `/marimo/yolo_code/datasets/levir_ship_yolo_seed42/`. Results are evidence, not paper-ready until seeds 43/44 confirm recovery.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, inspect, shutil, gc
from copy import deepcopy
from contextlib import contextmanager

ROOT = Path("/marimo/yolo_code").resolve()
assert ROOT.is_dir(), f"Sync code to Marimo first: {ROOT}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA_YAML = ROOT / "datasets/levir_ship_yolo_seed42/levir_ship.yaml"
OUT = ROOT / "runs/levir_h2_int8_rootcause_seed42_v1"
assert OUT.parent.resolve() == (ROOT / "runs").resolve()
HF_DATASET = "TLMHoang/FTSC_results"
HF_REVISION = "81edf3de222a65d3bb82" + "5036e61c1def9c5684ea"
SOURCE = "LEVIR-Ship/historical_reconstructions/RECON_H2_V1/seed_42/weights/best.pt"
EXPECTED_SHA = "e658614e617378110a280fe62927eb" + "0df6f708162a5117865fe45efe66612805"

DEVICE = "cuda:0"
WORKERS = 2
RUN_GPU = False  # deliberately SAFE; set True only after reviewing preflight
RUN_VARIANTS = ["legacy_minmax", "stable_minmax", "stable_entropy"]
# After the initial 3, append "stable_minmax_matched" and "stable_entropy_matched".
ARM = {
    "legacy_minmax": dict(buffer=False, entropy=False, geometry=False),
    "stable_minmax": dict(buffer=True, entropy=False, geometry=False),
    "stable_entropy": dict(buffer=True, entropy=True, geometry=False),
    "stable_minmax_matched": dict(buffer=True, entropy=False, geometry=True),
    "stable_entropy_matched": dict(buffer=True, entropy=True, geometry=True),
}
assert RUN_VARIANTS and len(RUN_VARIANTS) == len(set(RUN_VARIANTS))
assert set(RUN_VARIANTS).issubset(ARM)
print("Selected arms:", RUN_VARIANTS)
print("Outputs:", OUT)
print("GPU enabled:", RUN_GPU)

## Preflight: preserve the published evaluation contract

Check that the code is from the intended fork; inspect the exact calibrator source before creating any engine. PT32/INT8 inference uses all 788 images with pixel-parity checking. The first PT32 reference should reproduce approximately 81.25 / 31.22 / 80.55 for seed 42; stop if it does not.

In [ ]:
from evaluate_test.standard_detection_metrics import ensure_local_ultralytics
ensure_local_ultralytics()
from utils import ftsc_hf_all_trt_int8_levir_fp16parity_v2 as v2
from utils import ftsc_hf_all_trt_int8_paper_v1 as common
from ultralytics.utils.export import engine as trt_engine

def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

SOURCE_FUNC = inspect.getsource(trt_engine.onnx2engine)
MINMAX = "else trt.CalibrationAlgoType.MINMAX_CALIBRATION"
BUFFER_OLD = (
    '                    im0s = next(self.data_iter)["img"] / 255.0\n'
    '                    im0s = im0s.to("cuda") if im0s.device.type == "cpu" else im0s\n'
    '                    return [int(im0s.data_ptr())]'
)
BUFFER_NEW = (
    '                    im0s = next(self.data_iter)["img"] / 255.0\n'
    '                    self.current_batch = im0s.to("cuda").contiguous()\n'
    '                    torch.cuda.synchronize(self.current_batch.device)\n'
    '                    return [int(self.current_batch.data_ptr())]'
)
assert SOURCE_FUNC.count(MINMAX) == 1, "Exporter changed: re-audit algorithm"
assert SOURCE_FUNC.count(BUFFER_OLD) == 1, "Exporter changed: re-audit buffer"
assert v2.ROLES == {"main": 544, "small": 512}
assert v2.DATASET == "LEVIR-Ship"
print("Calibration implementation SHA256:", hashlib.sha256(SOURCE_FUNC.encode()).hexdigest())
print("Calibrator source contracts OK; input parity remains enforced by V2.")

In [ ]:
@contextmanager
def patched_export(arm, role):
    """In-memory only: never edit the canonical exporter.py or engine.py on disk."""
    import ultralytics.utils.export.engine as engine
    import ultralytics.engine.exporter as exporter

    old_engine = engine.onnx2engine
    old_loader = exporter.Exporter.get_int8_calibration_dataloader
    source = inspect.getsource(old_engine)
    assert source.count(MINMAX) == source.count(BUFFER_OLD) == 1
    new_source = source
    if arm["entropy"]:
        new_source = new_source.replace(
            MINMAX, "else trt.CalibrationAlgoType.ENTROPY_CALIBRATION_2", 1)
    if arm["buffer"]:
        new_source = new_source.replace(BUFFER_OLD, BUFFER_NEW, 1)
    if new_source != source:
        namespace = {}
        exec(compile(new_source, inspect.getsourcefile(old_engine) or "<probe>", "exec"),
             engine.__dict__, namespace)
        assert "onnx2engine" in namespace
        engine.onnx2engine = namespace["onnx2engine"]

    if arm["geometry"] and role == "main":
        def matched_train_loader(self, prefix=""):
            """Full TRAIN images: 512 original content + 16 pixels of 114 padding."""
            import torch
            import torch.nn.functional as F
            from ultralytics.data import build_yolo_dataset, build_dataloader
            from ultralytics.data.utils import check_det_dataset
            assert self.args.int8 and self.args.batch == 1 and max(self.imgsz) == 544
            assert self.args.split == "train" and self.args.fraction == 1.0
            cfg = deepcopy(self.args)
            cfg.imgsz = 512
            cfg.rect = False
            dataset = build_yolo_dataset(
                cfg, check_det_dataset(self.args.data)["train"], 1,
                check_det_dataset(self.args.data),
                mode="val", rect=False, stride=32, fraction=1.0)
            assert len(dataset) == 2320, "Expected 2320 LEVIR train images"
            loader = build_dataloader(dataset, batch=1, workers=0, drop_last=True)

            class PadTrain:
                batch_size = 1
                def __len__(self):
                    return len(loader)
                def __iter__(self):
                    for batch in loader:
                        x = batch["img"]
                        assert x.dtype == torch.uint8 and tuple(x.shape) == (1, 3, 512, 512)
                        y = F.pad(x, (16, 16, 16, 16), value=114)
                        assert tuple(y.shape) == (1, 3, 544, 544)
                        yield {"img": y}
            return PadTrain()
        exporter.Exporter.get_int8_calibration_dataloader = matched_train_loader
    try:
        yield
    finally:
        engine.onnx2engine = old_engine
        exporter.Exporter.get_int8_calibration_dataloader = old_loader

print("In-memory diagnostic patches defined. Production exporter remains unchanged.")

## Freeze checkpoint and a single PT32 reference

The checkpoint is read from a **pinned HF commit**, verified by SHA256, and evaluated via the unchanged V2 code. The reference is evaluated exactly once, then reused for all arms. If any output directory is partially written, the notebook **fails closed**, rather than resuming a potentially mismatched engine.

In [ ]:
def get_source():
    from huggingface_hub import hf_hub_download
    source = Path(hf_hub_download(
        repo_id=HF_DATASET, repo_type="dataset", filename=SOURCE,
        revision=HF_REVISION, cache_dir=str(OUT / "hf_readonly_cache")))
    assert sha(source) == EXPECTED_SHA, "Incorrect model checksum"
    return source

def make_reference(checkpoint, images):
    base = OUT / "reference"
    report = base / "pt32_reference_complete.json"
    if report.exists():
        row = json.loads(report.read_text())
        assert row["checkpoint_sha256"] == EXPECTED_SHA
        assert row["data_yaml_sha256"] == sha(DATA_YAML)
        for name in ("test_val_preprocessed_inputs_sha256.json", "small_inputs.json",
                     "test_size_ground_truth.json"):
            assert (base / "pt32_reference" / name).is_file()
        return row
    assert not base.exists(), "Partial reference; inspect before rerun or select new OUT"
    base.mkdir(parents=True)
    pt_main, main_pixels = v2.evaluate_main_pt32(
        checkpoint, DATA_YAML, images, base, DEVICE, WORKERS)
    pt_small, small_pixels = v2.predict_small(
        checkpoint, images, base, DEVICE, WORKERS, backend="pt32")
    keys = v2.METRIC_KEYS
    assert main_pixels["preprocessed_input_count"] == 788
    assert small_pixels["input_count"] == 788
    # Do not proceed if the control itself has broken parity with historical H2.
    assert abs(pt_main[keys[0]] - 0.8125051194881348) < 0.005
    assert abs(pt_main[keys[1]] - 0.3122204316803089) < 0.005
    assert abs(pt_small[keys[2]] - 0.8055214972426346) < 0.01
    row = {"checkpoint_sha256": EXPECTED_SHA, "data_yaml_sha256": sha(DATA_YAML),
           "main": pt_main, "small": pt_small,
           "main_input_sha256": main_pixels["preprocessed_input_manifest_sha256"],
           "small_input_sha256": small_pixels["input_manifest_sha256"]}
    report.write_text(json.dumps(row, indent=2, sort_keys=True) + "\n")
    return row

print("PT32 reference helpers defined; HF download happens only after RUN_GPU=True.")

## Build, evaluate and archive isolated variants

**Note:** Each arm calibrates on all 2,320 training images *twice* (MAIN 544 and SMALL 512) at batch1; this is a substantial GPU task. Start with **legacy_minmax → stable_minmax → stable_entropy**; only then add the geometry arms.

For every arm, the runner stores source SHA, calibration factors, engine SHA, AP50/mAP50:95/AP50-small, and deltas versus the *same* in-run PT32 reference. It checks 788-image parity and ground truth for each arm. No uploads.

In [ ]:
def execute_arm(name, checkpoint, images, calibration, code_hashes, runtime, reference):
    factors = ARM[name]
    work = OUT / "variants" / name
    report = work / "metrics.json"
    identity = {
        "source_sha256": EXPECTED_SHA,
        "yaml_sha256": sha(DATA_YAML),
        "calibration_train": calibration,
        "code_sha256": code_hashes,
        "runtime": runtime,
        "arm": factors,
        "calibrator_source_sha256": hashlib.sha256(SOURCE_FUNC.encode()).hexdigest(),
    }
    if report.exists():
        old = json.loads(report.read_text())
        assert old["identity"] == identity, "Mismatched cached arm provenance"
        for role in ("main", "small"):
            assert sha(work / "paper_export" / role / "source.engine") == old["engine_sha256"][role]
            assert sha(work / "paper_export" / role / "source.cache") == old["cache_sha256"][role]
        return old
    assert not work.exists(), f"Partial arm exists (manual inspection needed): {work}"
    work.mkdir(parents=True)
    (work / "pt32_reference").mkdir()
    root_pt = OUT / "reference/pt32_reference"
    for name_of_file in ("test_val_preprocessed_inputs_sha256.json",
                         "small_inputs.json", "test_size_ground_truth.json"):
        shutil.copy2(root_pt / name_of_file, work / "pt32_reference" / name_of_file)
    engines = {}
    for role in ("main", "small"):
        with patched_export(factors, role):
            engine_file, spec = v2.static_engine_for(
                checkpoint, EXPECTED_SHA, work, role, DATA_YAML,
                calibration, code_hashes, DEVICE, runtime)
        cache = work / "paper_export" / role / "source.cache"
        assert cache.is_file(), f"Calibration cache missing: {cache}"
        actual_header = cache.open("rb").readline()
        expected_calibration = b"Entropy" if factors["entropy"] else b"MinMax"
        assert expected_calibration in actual_header, (
            f"Wrong TRT calibration algorithm for {name}/{role}: {actual_header!r}")
        engines[role] = (engine_file, spec)
    main, main_inputs = v2.evaluate_main_int8(
        engines["main"][0], DATA_YAML, images, work, DEVICE, WORKERS,
        {"preprocessed_input_manifest_sha256": reference["main_input_sha256"]})
    small, small_inputs = v2.predict_small(
        engines["small"][0], images, work, DEVICE, WORKERS, backend="trt_int8")
    parity = v2.verify_small_parity(
        work,
        {"input_manifest_sha256": reference["small_input_sha256"],
         "input_count": 788,
         "gt_sha256": sha(root_pt / "test_size_ground_truth.json")},
        small_inputs)
    assert parity["main_and_small_input_pixel_parity"]
    keys = v2.METRIC_KEYS
    names = ("AP50", "mAP50_95", "AP50_small")
    actual = (main[keys[0]], main[keys[1]], small[keys[2]])
    control = (reference["main"][keys[0]], reference["main"][keys[1]],
               reference["small"][keys[2]])
    assert all(0 <= x <= 1 for x in actual + control)
    record = {
        "case": name, "identity": identity,
        "metric_pct": {k: round(100*x, 8) for k,x in zip(names, actual)},
        "pt32_pct": {k: round(100*x, 8) for k,x in zip(names, control)},
        "delta_pp": {k: round(100*(a-b), 8) for k,a,b in zip(names, actual, control)},
        "engine_sha256": {role: data[1]["engine_sha256"] for role,data in engines.items()},
        "cache_sha256": {role: sha(work / "paper_export" / role / "source.cache")
                         for role in engines},
        "main_and_small_input_pixel_parity": True,
    }
    assert not report.exists()
    report.write_text(json.dumps(record, indent=2, sort_keys=True) + "\n")
    return record

def compare_existing():
    rows = []
    for key in ARM:
        saved = OUT / "variants" / key / "metrics.json"
        if not saved.exists():
            continue
        a = json.loads(saved.read_text())
        rows.append({"case": key, **a["metric_pct"],
                     "delta_AP50_pp": a["delta_pp"]["AP50"]})
    if rows:
        from pandas import DataFrame
        display(DataFrame(rows))
    else:
        print("No experiments completed yet.")
    return rows

print("Variant evaluator defined; no engines built yet.")

In [ ]:
if RUN_GPU:
    import torch
    import tensorrt as trt
    assert torch.cuda.is_available(), "CUDA required"
    assert trt.__version__.split(".")[0] == "10", "This probe is for TRT 10 calibrator"
    assert DATA_YAML.exists(), f"Prepare frozen dataset: {DATA_YAML}"
    # One notebook/session per output root; refuse concurrent jobs.
    OUT.mkdir(parents=True, exist_ok=True)
    lock = OUT / "probe.lock"
    with lock.open("x") as f:
        f.write(f"PID={os.getpid()}\n")
    try:
        checkpoint = get_source()
        dataset_args = type("Args", (), {"data_yaml": DATA_YAML})()
        _, images, context, calibration = v2.dataset_context(dataset_args)
        assert len(images) == 788 and calibration["calibration_images"] == 2320
        code_hashes = v2.source_hashes()
        runtime = common.runtime_identity(DEVICE)
        reference = make_reference(checkpoint, images)
        for case in RUN_VARIANTS:
            print(f"--- INT8 LEVIR H2 seed42: {case} ---", flush=True)
            metrics = execute_arm(case, checkpoint, images, calibration,
                                  code_hashes, runtime, reference)
            print(json.dumps({"case": case, "metric_pct": metrics["metric_pct"],
                              "delta_pp": metrics["delta_pp"]}, indent=2), flush=True)
            gc.collect()
            torch.cuda.empty_cache()
    finally:
        lock.unlink(missing_ok=True)
else:
    print("PREVIEW ONLY — Set RUN_GPU=True after checking source/dataset/GPU availability.")

## Inspect differences and decide

A successful variant must recover accuracy on the **same seed 42** using the same checkpoint, test inputs, NMS and evaluator. If changing the buffer lifetime improves results, fix the underlying calibrator before experimenting with algorithm/geometry. If neither change helps, next step is raw-head tensor comparison / precision constraints for detection head + DFL. Do **not** overwrite previous `int8_static_fp16parity_v2_callfix` results or update final paper figures from this notebook alone.

In [ ]:
comparison = compare_existing()
if comparison:
    target = OUT / "rootcause_probe_summary.json"
    import pandas as pd
    table = pd.DataFrame(comparison)
    table.to_json(target, orient="records", indent=2)
    print("Saved comparison:", target)
    print("Paper promotion: blocked until causality is isolated and seeds 43/44 verified.")